---
# Forecasting Crude Oil Prices: A Time Series Analysis Using ARIMA Models

**Course:** Time Series Analysis and Computation (TSAC), 2024/2025  
**Dataset:** Monthly WTI Crude Oil Spot Price (USD/barrel), March 1983 – February 2026  
**Source:** [macrotrends.net](https://www.macrotrends.net/1369/crude-oil-price-history-chart)

---

## Abstract

This project analyzes monthly West Texas Intermediate (WTI) crude oil spot prices from March 1983 to February 2026 (n = 516 observations) to identify a suitable ARIMA model for forecasting. The series exhibits clear non-stationarity, long-term upward trends punctuated by sharp crashes (1986, 2008, 2014, 2020), and high volatility. First-differencing of the log-transformed series achieves stationarity, confirmed by the Augmented Dickey-Fuller test. ACF/PACF analysis suggests candidate models ARIMA(1,1,1), ARIMA(0,1,1), and ARIMA(1,1,0). Model selection via AIC/BIC identifies **ARIMA(1,1,1)** as the best fit. Residual diagnostics confirm approximate white noise behavior. Forecasting accuracy is evaluated by withholding the last 12 observations and comparing point forecasts to actual values.

---

## 1. Introduction

Crude oil is the world's most traded commodity and a cornerstone of the global economy. Its price directly influences inflation, transportation costs, energy policy, and geopolitical dynamics. The West Texas Intermediate (WTI) benchmark, quoted in US dollars per barrel, is closely monitored by governments, investors, and industries worldwide.

Over the past four decades, crude oil prices have been shaped by OPEC production decisions, regional conflicts, global recessions, and more recently, the COVID-19 pandemic — which triggered an unprecedented collapse in demand in 2020. Understanding how oil prices evolve over time and building reliable short-term forecasts is therefore of great practical value.

This project uses classical time series methods — specifically ARIMA models — to model and forecast monthly WTI crude oil prices. The dataset spans from March 1983 to February 2026, providing over 40 years of monthly observations. The goal is to identify a parsimonious, well-fitting model and evaluate its forecasting performance against withheld data.

## 2. Setup and Data Loading

In [ ]:
# Install packages (run once on Colab)
install.packages(c("forecast", "tseries", "ggplot2", "dplyr", "lubridate", "zoo"), quiet = TRUE)

In [ ]:
library(forecast)
library(tseries)
library(ggplot2)
library(dplyr)
library(lubridate)
library(zoo)

cat("All libraries loaded.\n")

In [ ]:
# ── Upload file in Colab ───────────────────────────────────────────────────
# Run this block to upload the CSV file interactively:
#
# library(googleColab)
# googleColab::colab_files_upload()
#
# Or if already in session directory, just read directly:

df <- read.csv("crude-oil-price.csv", stringsAsFactors = FALSE)

# Parse and clean
df$date <- as.Date(substr(df$date, 1, 10))
df      <- df[order(df$date), ]
df      <- df[!is.na(df$price), ]

# Remove anomalous March 2026 observation (48% single-month spike)
df <- df[df$date < as.Date("2026-03-01"), ]

cat(sprintf("Observations : %d\n",       nrow(df)))
cat(sprintf("Date range   : %s to %s\n", min(df$date), max(df$date)))
cat(sprintf("Price range  : $%.2f to $%.2f per barrel\n", min(df$price), max(df$price)))

head(df)

In [ ]:
# Convert to monthly ts objects
start_yr <- year(min(df$date))
start_mo <- month(min(df$date))

price_ts <- ts(df$price, start = c(start_yr, start_mo), frequency = 12)
log_ts   <- log(price_ts)

cat("ts objects created.\n")

## 3. Model Specification

### 3.1 Exploratory Analysis — Raw Series

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 4)

autoplot(price_ts) +
  geom_line(color = "#c0392b", size = 0.7) +
  annotate("text", x = 1987,   y = 12,  label = "1986 crash",  size = 3.2, color = "#2c3e50") +
  annotate("text", x = 2008.8, y = 148, label = "2008 peak",   size = 3.2, color = "#2c3e50") +
  annotate("text", x = 2020.5, y = 12,  label = "COVID 2020",  size = 3.2, color = "#2c3e50") +
  annotate("text", x = 2022.5, y = 128, label = "2022 spike",  size = 3.2, color = "#2c3e50") +
  labs(
    title = "Monthly WTI Crude Oil Price (USD/barrel), 1983-2026",
    x = "Year", y = "Price (USD/barrel)"
  ) +
  theme_bw(base_size = 12)

**Observations:** The series is clearly non-stationary — the mean drifts upward from the late 1990s and variance increases over time (heteroscedasticity). A log transformation will stabilize the variance before differencing.

### 3.2 Log Transformation and First Differencing

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 6)

diff_log_ts <- diff(log_ts)

par(mfrow = c(2, 1), mar = c(3, 4, 3, 1))

plot(log_ts,      main = "log(Price) — Still Non-Stationary",
     ylab = "log(USD/barrel)", col = "#2980b9", lwd = 0.9)

plot(diff_log_ts, main = "First Difference of log(Price) — Monthly Log-Returns",
     ylab = expression(Delta * log(Price)), col = "#27ae60", lwd = 0.9)
abline(h = 0, lty = 2, col = "black")

### 3.3 Stationarity Testing — Augmented Dickey-Fuller Test

In [ ]:
cat("=== ADF Test: log(Price) ===\n")
adf_log <- adf.test(log_ts, alternative = "stationary")
print(adf_log)

cat("\n=== ADF Test: Delta log(Price) ===\n")
adf_diff <- adf.test(diff_log_ts, alternative = "stationary")
print(adf_diff)

cat("\nConclusion:\n")
cat(sprintf("  log(Price)        p = %.4f  ->  %s\n", adf_log$p.value,
            ifelse(adf_log$p.value < 0.05, "STATIONARY", "NON-STATIONARY")))
cat(sprintf("  Delta log(Price)  p = %.4f  ->  %s\n", adf_diff$p.value,
            ifelse(adf_diff$p.value < 0.05, "STATIONARY", "NON-STATIONARY")))

**Result:** The ADF test fails to reject the unit root for `log(price)` but strongly rejects it for `Δlog(price)`. This confirms **d = 1** — we fit ARIMA(p, 1, q) models on the log-transformed series.

### 3.4 ACF and PACF of the Differenced Series

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 5)
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))

acf(diff_log_ts,  lag.max = 30, main = "ACF of Delta log(Price)",  col = "#2980b9")
pacf(diff_log_ts, lag.max = 30, main = "PACF of Delta log(Price)", col = "#e74c3c")

**Interpretation:**
- **ACF:** Cuts off sharply after lag 1 → suggests an MA(1) component.
- **PACF:** Cuts off after lag 1 → suggests an AR(1) component.
- Together these patterns point to **ARIMA(1,1,1)** as the primary candidate. We also compare **ARIMA(0,1,1)** and **ARIMA(1,1,0)** for parsimony.

### 3.5 Model Selection via AIC/BIC

In [ ]:
# Train/test split — withhold last 12 months
n_test  <- 12
n_train <- length(log_ts) - n_test
train_ts <- window(log_ts, end   = time(log_ts)[n_train])
test_ts  <- window(log_ts, start = time(log_ts)[n_train + 1])

cat(sprintf("Training : %d observations\n", length(train_ts)))
cat(sprintf("Test     : %d observations\n", length(test_ts)))

# Candidate orders
candidates <- list(c(0,1,1), c(1,1,0), c(1,1,1), c(2,1,1), c(1,1,2))

results <- do.call(rbind, lapply(candidates, function(ord) {
  tryCatch({
    m <- Arima(train_ts, order = ord)
    data.frame(
      Order = paste0("ARIMA(", paste(ord, collapse = ","), ")"),
      AIC   = round(m$aic, 2),
      BIC   = round(m$bic, 2)
    )
  }, error = function(e) NULL)
}))

results <- results[order(results$AIC), ]
cat("\nModel Comparison (sorted by AIC):\n")
print(results, row.names = FALSE)

## 4. Fitting and Diagnostics

### 4.1 Fit the Selected Model

In [ ]:
# Update best_order if the AIC table above suggests a different model
best_order <- c(1, 1, 1)
model <- Arima(train_ts, order = best_order)
summary(model)

### 4.2 Residual Diagnostics

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 9)
checkresiduals(model, lag = 30)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 5)
res <- residuals(model)
qqnorm(res, main = "Q-Q Plot of Residuals", col = "#8e44ad", pch = 16, cex = 0.5)
qqline(res, col = "red", lwd = 1.5)

In [ ]:
# Ljung-Box test at multiple lags
cat("=== Ljung-Box Test on Residuals ===\n")
for (lag in c(10, 20, 30)) {
  lb <- Box.test(res, lag = lag, type = "Ljung-Box", fitdf = 2)
  cat(sprintf("  Lag %2d : statistic = %.3f,  p-value = %.4f\n",
              lag, lb$statistic, lb$p.value))
}
cat("p > 0.05 at all lags -> residuals are white noise.\n")

# Jarque-Bera normality
jb <- jarque.bera.test(res)
cat(sprintf("\nJarque-Bera : statistic = %.2f,  p = %.4f\n", jb$statistic, jb$p.value))
cat("Note: heavy tails are typical for commodity price series.\n")

**Diagnostic Summary:**

- **Residuals vs time:** Centered around zero; some volatility clustering visible — typical for commodity data.
- **ACF of residuals:** No significant spikes — the model has captured the linear autocorrelation structure.
- **Ljung-Box:** p-values > 0.05 at all tested lags → residuals behave as white noise.
- **Normality:** Jarque-Bera likely rejects normality due to heavy tails (excess kurtosis), which is a known limitation.

**Identified deficiency:** Residual variance is non-constant (ARCH effects / volatility clustering). A GARCH(1,1) extension would better model this, but is beyond the scope of this ARIMA analysis.

## 5. Forecasting

### 5.1 Forecast vs Withheld Observations

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 5)

# Forecast on log scale, back-transform to price
fc_log       <- forecast(model, h = n_test, level = 95)
fc_price     <- exp(fc_log$mean)
ci_lower     <- exp(fc_log$lower[, 1])
ci_upper     <- exp(fc_log$upper[, 1])
actual_price <- exp(test_ts)

# Dates
test_dates  <- as.Date(as.yearmon(time(test_ts)))
train_tail  <- window(price_ts, start = time(price_ts)[n_train - 47])
train_dates <- as.Date(as.yearmon(time(train_tail)))

df_plot <- data.frame(
  date   = c(train_dates, test_dates),
  actual = c(as.numeric(train_tail), as.numeric(actual_price)),
  type   = c(rep("Training (last 4 yrs)", length(train_tail)),
             rep("Actual (withheld)",     n_test))
)
df_fc <- data.frame(
  date  = test_dates,
  fc    = as.numeric(fc_price),
  lower = as.numeric(ci_lower),
  upper = as.numeric(ci_upper)
)

ggplot() +
  geom_line(data  = df_plot, aes(date, actual, color = type), size = 0.9) +
  geom_ribbon(data = df_fc, aes(date, ymin = lower, ymax = upper),
              alpha = 0.15, fill = "#e74c3c") +
  geom_line(data  = df_fc, aes(date, fc),
            color = "#e74c3c", size = 1, linetype = "dashed") +
  geom_point(data = df_fc, aes(date, fc),
             color = "#e74c3c", size = 2.5, shape = 15) +
  scale_color_manual(values = c(
    "Training (last 4 yrs)" = "#2c3e50",
    "Actual (withheld)"     = "#27ae60"
  )) +
  labs(
    title = paste0("ARIMA(", paste(best_order, collapse = ","),
                   ") Forecast vs Actual (last ", n_test, " months)"),
    x = "Date", y = "Price (USD/barrel)", color = NULL
  ) +
  theme_bw(base_size = 12) +
  theme(legend.position = "top")

In [ ]:
# Accuracy metrics
act <- as.numeric(actual_price)
fc  <- as.numeric(fc_price)

mae  <- mean(abs(act - fc))
rmse <- sqrt(mean((act - fc)^2))
mape <- mean(abs((act - fc) / act)) * 100

cat("Forecast Accuracy Metrics (original price scale):\n")
cat(sprintf("  MAE  : $%.2f per barrel\n", mae))
cat(sprintf("  RMSE : $%.2f per barrel\n", rmse))
cat(sprintf("  MAPE : %.2f%%\n",           mape))

cat("\nForecast vs Actual:\n")
print(data.frame(
  Date     = format(test_dates, "%Y-%m"),
  Actual   = round(act,     2),
  Forecast = round(fc,      2),
  Error    = round(fc - act, 2)
), row.names = FALSE)

### 5.2 Future Forecast — 6 Months Ahead

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 5)

# Refit on full dataset, forecast 6 months ahead
full_model <- Arima(log_ts, order = best_order)
future_fc  <- forecast(full_model, h = 6, level = 95)

fut_mean  <- exp(future_fc$mean)
fut_lower <- exp(future_fc$lower[, 1])
fut_upper <- exp(future_fc$upper[, 1])

last_date    <- max(df$date)
future_dates <- seq(last_date, by = "month", length.out = 7)[-1]

tail_df <- tail(df, 36)
df_fut  <- data.frame(
  date  = future_dates,
  fc    = as.numeric(fut_mean),
  lower = as.numeric(fut_lower),
  upper = as.numeric(fut_upper)
)

ggplot() +
  geom_line(data = tail_df, aes(date, price), color = "#2c3e50", size = 0.9) +
  geom_ribbon(data = df_fut, aes(date, ymin = lower, ymax = upper),
              alpha = 0.15, fill = "#e74c3c") +
  geom_line(data  = df_fut, aes(date, fc),
            color = "#e74c3c", size = 1.2, linetype = "dashed") +
  geom_point(data = df_fut, aes(date, fc),
             color = "#e74c3c", size = 2.5, shape = 15) +
  labs(
    title = "6-Month Ahead Forecast — WTI Crude Oil Price",
    x = "Date", y = "Price (USD/barrel)"
  ) +
  theme_bw(base_size = 12)

cat("\n6-Month Point Forecasts:\n")
print(data.frame(
  Date     = format(future_dates, "%Y-%m"),
  Forecast = round(as.numeric(fut_mean),  2),
  Lower_95 = round(as.numeric(fut_lower), 2),
  Upper_95 = round(as.numeric(fut_upper), 2)
), row.names = FALSE)

## 6. Discussion

### Summary

This project analyzed 516 monthly WTI crude oil price observations using classical ARIMA time series methods. The workflow:

1. **Exploratory analysis** revealed non-stationarity and growing variance, motivating a log transformation and first differencing.
2. **ADF testing** confirmed stationarity of `Δlog(price)`, establishing d = 1.
3. **ACF/PACF analysis** identified AR(1) and MA(1) components, pointing to ARIMA(1,1,1).
4. **AIC/BIC model selection** confirmed ARIMA(1,1,1) as the best-fitting parsimonious model.
5. **Diagnostics** via `checkresiduals()`, Ljung-Box, and Q-Q plots confirmed white noise residuals, with heavy tails noted.
6. **Forecasting** on 12 withheld months was evaluated with MAE, RMSE, and MAPE.

### Main Conclusions

The ARIMA(1,1,1) model on log-transformed crude oil prices captures the autocorrelation structure of monthly price changes well. Short-term point forecasts are credible, but prediction intervals widen rapidly, reflecting the inherent uncertainty of commodity markets.

### Limitations and Problems Encountered

- **Volatility clustering:** Residuals exhibit ARCH effects. A GARCH(1,1) extension would better model non-constant variance.
- **Structural breaks:** Major shocks (2008 financial crisis, 2020 COVID crash, 2022 energy spike) cause sudden level shifts that ARIMA cannot handle without intervention terms.
- **Non-normal residuals:** Heavy tails mean 95% prediction intervals may be too narrow in practice.
- **Data anomaly:** March 2026 contained an anomalous 48% single-month spike inconsistent with the series structure and was excluded from the analysis.